<a href="https://colab.research.google.com/github/adelekeadeniyan/ibadan-solar-forecasting/blob/main/notebooks/ibadan_solar_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import glob
import os

# 1. Locate and Load NASA POWER CSV
# Updated to load specific file from Google Drive
target_file = '/content/drive/My Drive/POWER_Point_Hourly_20230101_20231130_007d38N_003d94E_LST.csv'

# Ensure the file exists before proceeding
if not os.path.exists(target_file):
    raise FileNotFoundError(f"The specified file was not found in Google Drive: {target_file}")

print(f"Reading dataset: {target_file}")

# NASA POWER files contain header metadata lines starting with -END HEADER-
header_row = 0
with open(target_file, "r") as f:
    for i, line in enumerate(f):
        if "-END HEADER-" in line:
            header_row = i + 1
            break

df = pd.read_csv(target_file, skiprows=header_row)
df.columns = [c.strip() for c in df.columns]

# Standardize column naming
rename_dict = {}
for col in df.columns:
    if "ALLSKY_SFC_SW_DWN" in col:
        rename_dict[col] = "Irradiance"
    elif "T2M" in col:
        rename_dict[col] = "Temperature"
    elif "RH2M" in col:
        rename_dict[col] = "Humidity"
    elif "HR" in col or "HOUR" in col:
        rename_dict[col] = "Hour"
    elif "MO" in col or "MONTH" in col:
        rename_dict[col] = "Month"
    elif "DY" in col or "DAY" in col:
        rename_dict[col] = "Day"

df.rename(columns=rename_dict, inplace=True)

# Replace NASA fill values (-999.0) with NaN and interpolate
df.replace(-999.0, np.nan, inplace=True)
df.interpolate(method="linear", inplace=True)
df.dropna(inplace=True)

# 2. Feature Engineering
# Cyclical encoding for solar diurnal cycles
df["Hour_Sin"] = np.sin(2 * np.pi * df["Hour"] / 24.0)
df["Hour_Cos"] = np.cos(2 * np.pi * df["Hour"] / 24.0)
df["Month_Sin"] = np.sin(2 * np.pi * df["Month"] / 12.0)
df["Month_Cos"] = np.cos(2 * np.pi * df["Month"] / 12.0)

# Historical lags
df["Lag_Irradiance_1h"] = df["Irradiance"].shift(1)
df["Lag_Temp_1h"] = df["Temperature"].shift(1)
df.dropna(inplace=True)

# 3. Exploratory Data Figures
sns.set_theme(style="whitegrid", font="sans-serif")

# Correlation Matrix
plt.figure(figsize=(8, 6))
corr_cols = ["Irradiance", "Temperature", "Humidity", "Hour", "Month"]
corr = df[corr_cols].corr()
sns.heatmap(corr, annot=True, cmap="Blues", fmt=".2f", square=True)
plt.title("Correlation Analysis of Surface Variables in Ibadan", fontsize=12)
plt.tight_layout()
plt.savefig("figure1_correlation_matrix.png", dpi=300)
plt.close()

# 4. Train-Test Partition (Chronological 80/20 split)
features = [
    "Temperature", "Humidity", "Hour_Sin", "Hour_Cos",
    "Month_Sin", "Month_Cos", "Lag_Irradiance_1h", "Lag_Temp_1h"
]
target = "Irradiance"

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

# 5. Model Initialization and Training
models = {
    "Multiple Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, max_depth=5, random_state=42)
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    preds = np.clip(preds, 0, None)  # Solar irradiance cannot be negative
    predictions[name] = preds

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results.append({"Model": name, "MAE": round(mae, 3), "RMSE": round(rmse, 3), "R2": round(r2, 4)})

results_table = pd.DataFrame(results)
print("\n=== Model Performance Evaluation ===")
print(results_table.to_string(index=False))

# 6. Prediction Trajectory Plot (72-Hour Sample)
sample_hours = 72
eval_slice = y_test.iloc[100:100 + sample_hours].reset_index(drop=True)

plt.figure(figsize=(11, 4.5))
plt.plot(eval_slice, label="Ground Truth", color="black", linewidth=2)
for name in ["Random Forest", "Gradient Boosting"]:
    pred_slice = predictions[name][100:100 + sample_hours]
    plt.plot(pred_slice, linestyle="--", label=f"Predicted ({name})")

plt.xlabel("Forecast Horizon (Hours)", fontsize=11)
plt.ylabel("Solar Irradiance (kW/m²)", fontsize=11)
plt.title("72-Hour Prediction Trajectory on Ibadan Test Interval", fontsize=12)
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig("figure2_actual_vs_predicted.png", dpi=300)
plt.close()

# 7. Feature Importance Plot
gbr = models["Gradient Boosting"]
importance = pd.Series(gbr.feature_importances_, index=features).sort_values()

plt.figure(figsize=(8, 4.5))
importance.plot(kind="barh", color="steelblue")
plt.xlabel("Relative Importance Score", fontsize=11)
plt.title("Gradient Boosting Feature Importance", fontsize=12)
plt.tight_layout()
plt.savefig("figure3_feature_importance.png", dpi=300)
plt.close()

# 8. Boosting Iteration Loss (Epoch Equivalent)
train_deviance = gbr.train_score_
plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(train_deviance) + 1), train_deviance, color="darkred")
plt.xlabel("Boosting Iteration (Estimators)", fontsize=11)
plt.ylabel("Training Deviance (Loss)", fontsize=11)
plt.title("Loss Reduction Across Boosting Iterations", fontsize=12)
plt.tight_layout()
plt.savefig("figure4_boosting_loss_curve.png", dpi=300)
plt.close()

print("\nProcessing complete. All manuscript figures have been exported.")

Reading dataset: /content/drive/My Drive/POWER_Point_Hourly_20230101_20231130_007d38N_003d94E_LST.csv

=== Model Performance Evaluation ===
                     Model    MAE   RMSE     R2
Multiple Linear Regression 33.417 53.930 0.9576
             Random Forest 17.796 33.996 0.9832
         Gradient Boosting 17.226 32.207 0.9849

Processing complete. All manuscript figures have been exported.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Check if df and results_table are defined, as they are expected from previous cells
if 'df' not in globals():
    raise NameError("DataFrame 'df' is not defined. Please ensure the preceding data loading and processing cells are executed first.")
if 'results_table' not in globals():
    raise NameError("DataFrame 'results_table' is not defined. Please ensure the preceding model training and evaluation cells are executed first.")

# 1. Descriptive Statistics Table (LaTeX & CSV)
desc_cols = ["Irradiance", "Temperature", "Humidity"]
desc_stats = df[desc_cols].describe().T[["count", "mean", "std", "min", "max"]]
desc_stats.columns = ["Sample Count", "Mean", "Std Dev", "Min", "Max"]
desc_stats = desc_stats.round(2)

print("\n--- Descriptive Statistics Table (Markdown) ---")
print(desc_stats.to_markdown())

# Export directly to LaTeX format for manuscript insertion
desc_stats.to_latex("table1_descriptive_stats.tex")
desc_stats.to_csv("table1_descriptive_stats.csv")

# 2. Export Model Performance Table
results_table.to_latex("table2_model_performance.tex", index=False)
results_table.to_csv("table2_model_performance.csv", index=False)
print("\n--- Model Performance Table (LaTeX) ---")
print(results_table.to_latex(index=False))

# 3. Generate Diurnal and Seasonal Irradiance Comparison (Dry vs. Rainy)
# In Ibadan, Jan-Mar represents peak dry season; Jun-Aug represents peak monsoon
dry_season = df[df["Month"].isin([1, 2, 3])].groupby("Hour")["Irradiance"].mean()
rainy_season = df[df["Month"].isin([6, 7, 8])].groupby("Hour")["Irradiance"].mean()

plt.figure(figsize=(8, 4.5))
plt.plot(dry_season.index, dry_season.values, label="Dry Season (Jan - Mar)", color="darkorange", linewidth=2.2)
plt.plot(rainy_season.index, rainy_season.values, label="Rainy Season (Jun - Aug)", color="navy", linewidth=2.2, linestyle="--")

plt.xlabel("Hour of Day (Local Solar Time)", fontsize=11)
plt.ylabel("Mean Solar Irradiance (W/m²)", fontsize=11)
plt.title("Diurnal Irradiance Profiles Across Distinct Seasons in Ibadan", fontsize=12)
plt.xticks(range(0, 24, 2))
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig("figure5_seasonal_diurnal_profiles.png", dpi=300)
plt.close()

print("Tables exported to .tex/.csv and Figure 5 generated successfully.")


--- Descriptive Statistics Table (Markdown) ---
|             |   Sample Count |   Mean |   Std Dev |   Min |    Max |
|:------------|---------------:|-------:|----------:|------:|-------:|
| Irradiance  |           8015 | 192.1  |    257.9  |  0    | 979.47 |
| Temperature |           8015 |  25.61 |      2.71 | 15.2  |  33.52 |
| Humidity    |           8015 |  87.22 |     12.86 | 34.13 | 100    |

--- Model Performance Table (LaTeX) ---
\begin{tabular}{lrrr}
\toprule
Model & MAE & RMSE & R2 \\
\midrule
Multiple Linear Regression & 33.417000 & 53.930000 & 0.957600 \\
Random Forest & 17.796000 & 33.996000 & 0.983200 \\
Gradient Boosting & 17.226000 & 32.207000 & 0.984900 \\
\bottomrule
\end{tabular}

Tables exported to .tex/.csv and Figure 5 generated successfully.


In [ ]:
import zipfile
import os

# List of files to be zipped
files_to_zip = [
    'figure1_correlation_matrix.png',
    'figure2_actual_vs_predicted.png',
    'figure3_feature_importance.png',
    'figure4_boosting_loss_curve.png',
    'figure5_seasonal_diurnal_profiles.png',
    'table1_descriptive_stats.tex',
    'table1_descriptive_stats.csv',
    'table2_model_performance.tex',
    'table2_model_performance.csv'
]

zip_filename = 'results.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        if os.path.exists(file):
            zipf.write(file, os.path.basename(file)) # Write with just the filename
            print(f'Added {file} to {zip_filename}')
        else:
            print(f'Warning: File not found and skipped: {file}')

print(f'Successfully created {zip_filename}. You can now download it.')


Added figure1_correlation_matrix.png to results.zip
Added figure2_actual_vs_predicted.png to results.zip
Added figure3_feature_importance.png to results.zip
Added figure4_boosting_loss_curve.png to results.zip
Added figure5_seasonal_diurnal_profiles.png to results.zip
Added table1_descriptive_stats.tex to results.zip
Added table1_descriptive_stats.csv to results.zip
Added table2_model_performance.tex to results.zip
Added table2_model_performance.csv to results.zip
Successfully created results.zip. You can now download it.
